In [1]:
%%capture
!pip install yt-dlp opencv-python mediapipe==0.10.14 easyocr ultralytics numpy tqdm

In [2]:
import cv2
import numpy as np
import mediapipe as mp
import easyocr
from ultralytics import YOLO
import yt_dlp
import os
import re
import unicodedata
from pathlib import Path
from tqdm import tqdm
import shutil
import gc
import warnings
import difflib

warnings.filterwarnings("ignore")

# =============================================================================
# KEYPOINT LAYOUT — 231 dims total (77 joints × 3 coords)
# [  0: 24] Body      —  8 joints × 3
# [ 24:105] Face      — 27 joints × 3
# [105:168] Left Hand — 21 joints × 3
# [168:231] Right Hand— 21 joints × 3
# =============================================================================

FACE_INDICES = [
    61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291, 375,
    33, 159, 133, 145,
    362, 386, 263, 374,
    70, 63, 105, 107,
    336, 296, 334,
]

DIMS_BODY  = 8  * 3   # 24
DIMS_FACE  = 27 * 3   # 81
DIMS_HAND  = 21 * 3   # 63
TOTAL_DIMS = DIMS_BODY + DIMS_FACE + DIMS_HAND * 2  # 231

IDX_BODY_START  = 0
IDX_FACE_START  = DIMS_BODY
IDX_LH_START    = DIMS_BODY + DIMS_FACE
IDX_RH_START    = DIMS_BODY + DIMS_FACE + DIMS_HAND


# =============================================================================
# TEXT NORMALIZATION & FUZZY MATCHING
# =============================================================================

# Các ký tự dễ bị OCR nhầm lẫn — map về dạng chuẩn
OCR_CONFUSION_MAP = {
    # Dấu câu bị nhầm
    ':': '.',
    ';': '.',
    '!': '.',
    # Ký tự số/chữ bị nhầm
    '0': 'o',
    '1': 'l',
    '|': 'l',
    # Khoảng trắng dư
    '\u00a0': ' ',   # non-breaking space
    '\t': ' ',
}

def normalize_text(text: str) -> str:
    """
    Chuẩn hóa text OCR để so sánh:
      1. Unicode NFC
      2. Lowercase
      3. Map các ký tự dễ nhầm về dạng chuẩn
      4. Collapse whitespace
      5. Bỏ dấu câu cuối câu
    """
    if not text:
        return ""

    # 1. Unicode normalize
    text = unicodedata.normalize("NFC", text)

    # 2. Lowercase
    text = text.lower()

    # 3. Map ký tự nhầm lẫn
    for wrong, right in OCR_CONFUSION_MAP.items():
        text = text.replace(wrong, right)

    # 4. Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # 5. Bỏ dấu câu cuối (. , : ; ! ?)
    text = re.sub(r'[.,:;!?]+$', '', text).strip()

    return text


def text_similarity(a: str, b: str) -> float:
    """
    Trả về similarity score [0.0, 1.0] giữa 2 text sau khi normalize.
    Dùng SequenceMatcher (giống difflib.get_close_matches).
    """
    na = normalize_text(a)
    nb = normalize_text(b)
    if not na and not nb:
        return 1.0
    if not na or not nb:
        return 0.0
    return difflib.SequenceMatcher(None, na, nb).ratio()


def is_same_sentence(current: str, new_text: str, threshold: float = 0.82) -> bool:
    """
    Trả về True nếu new_text là cùng câu với current
    (OCR nhận lại cùng nội dung nhưng có lỗi nhỏ).

    threshold=0.82: thực nghiệm tốt cho tiếng Việt với OCR noise.
    Tăng lên 0.90 nếu muốn strict hơn.
    """
    if not current or not new_text:
        return False
    sim = text_similarity(current, new_text)
    return sim >= threshold


def pick_better_text(a: str, b: str) -> str:
    """
    Trong 2 phiên bản OCR của cùng 1 câu, chọn bản "sạch" hơn:
      - Ưu tiên bản dài hơn (thường đầy đủ hơn)
      - Nếu bằng nhau, giữ bản cũ (a)
    """
    return b if len(b.strip()) > len(a.strip()) else a


# =============================================================================
# PROCESSOR
# =============================================================================

class YouTubeToSignSAMProcessor:
    def __init__(self, output_dataset="YOUTUBE_SIGN", fps=30, ocr_interval=8,
                 similarity_threshold=0.82):
        self.output_dataset       = output_dataset
        self.fps                  = fps
        self.ocr_interval         = ocr_interval
        self.similarity_threshold = similarity_threshold

        self.base_dir   = Path("/kaggle/working/dataset") / output_dataset
        self.video_dir  = Path("/kaggle/working/temp_videos")
        self.motion_dir = self.base_dir / "new_joints"
        self.text_dir   = self.base_dir / "texts"

        for d in [self.video_dir, self.motion_dir, self.text_dir]:
            d.mkdir(parents=True, exist_ok=True)

        print("📥 Đang khởi tạo models...")

        self.mp_pose = mp.solutions.pose.Pose(
            static_image_mode=False, model_complexity=1,
            min_detection_confidence=0.5, min_tracking_confidence=0.5
        )
        self.mp_hands = mp.solutions.hands.Hands(
            static_image_mode=False, max_num_hands=2,
            min_detection_confidence=0.5, min_tracking_confidence=0.5
        )
        self.mp_face = mp.solutions.face_mesh.FaceMesh(
            static_image_mode=False, max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.5, min_tracking_confidence=0.5
        )

        print(" - Loading EasyOCR...")
        try:
            self.reader = easyocr.Reader(['vi', 'en'], gpu=True)
        except Exception:
            self.reader = easyocr.Reader(['vi', 'en'], gpu=False)

        print(" - Loading YOLO...")
        self.yolo = YOLO('yolov8n.pt')

        self.all_file_names = []
        print(f"✅ Khởi tạo hoàn tất! Vector size = {TOTAL_DIMS} dims")
        print(f"   Similarity threshold = {self.similarity_threshold}")

    # =========================================================================
    # DOWNLOAD
    # =========================================================================

    def download_playlist(self, playlist_url):
        print(f"📥 Đang tải playlist: {playlist_url}")
        ydl_opts = {
            'format': 'best[ext=mp4]/best',
            'outtmpl': str(self.video_dir / '%(playlist_index)03d_%(id)s.%(ext)s'),
            'quiet': True, 'ignoreerrors': True, 'no_warnings': True
        }
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([playlist_url])
        except Exception as e:
            print(f"⚠️ {e}")

        video_files = []
        for ext in ['*.mp4', '*.mkv', '*.webm']:
            video_files.extend(list(self.video_dir.glob(ext)))
        video_files = sorted(video_files)
        print(f"✅ Đã tìm thấy {len(video_files)} video.")
        return video_files

    # =========================================================================
    # TEXT DETECTION
    # =========================================================================

    def detect_text_regions(self, frame):
        results = self.reader.readtext(frame)
        subtitle_texts, other_texts = [], []
        h, w, _ = frame.shape
        SUBTITLE_THRESHOLD = h * 0.75

        for (bbox, text, conf) in results:
            if conf < 0.35:
                continue
            clean = text.strip()
            if not clean or "PARD" in clean.upper():
                continue
            y_center = (bbox[0][1] + bbox[2][1]) / 2
            (subtitle_texts if y_center > SUBTITLE_THRESHOLD else other_texts).append(clean)

        if subtitle_texts:
            return " ".join(subtitle_texts)
        elif other_texts:
            return " ".join(other_texts)
        return ""

    # =========================================================================
    # KEYPOINT EXTRACTION — 231 dims (giữ nguyên)
    # =========================================================================

    def extract_openpose_keypoints(self, frame):
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pose_results  = self.mp_pose.process(frame_rgb)
        hands_results = self.mp_hands.process(frame_rgb)
        face_results  = self.mp_face.process(frame_rgb)

        flags = {'pose_detected': False, 'face_detected': False,
                 'lh_detected': False, 'rh_detected': False}

        # --- BODY ---
        body_kp = []
        if pose_results.pose_landmarks:
            flags['pose_detected'] = True
            lm = pose_results.pose_landmarks.landmark
            left_hip, right_hip = lm[23], lm[24]
            cx = (left_hip.x + right_hip.x) / 2
            cy = (left_hip.y + right_hip.y) / 2
            left_shoulder, right_shoulder = lm[11], lm[12]
            sh_cx = (left_shoulder.x + right_shoulder.x) / 2
            sh_cy = (left_shoulder.y + right_shoulder.y) / 2
            torso = np.sqrt((sh_cx - cx)**2 + (sh_cy - cy)**2)
            scale = torso if torso > 1e-5 else 1.0

            def norm(lmk):
                return [(lmk.x - cx) / scale, (lmk.y - cy) / scale, lmk.visibility]

            body_kp.extend(norm(lm[0]))
            body_kp.extend([(sh_cx - cx)/scale, (sh_cy - cy)/scale,
                             (left_shoulder.visibility + right_shoulder.visibility)/2])
            for idx in [12, 14, 16, 11, 13, 15]:
                body_kp.extend(norm(lm[idx]))

            self._last_cx, self._last_cy, self._last_scale = cx, cy, scale
            self._last_nose = ((lm[0].x - cx)/scale, (lm[0].y - cy)/scale)
            self._last_rw_n = ((lm[16].x - cx)/scale, (lm[16].y - cy)/scale)
            self._last_lw_n = ((lm[15].x - cx)/scale, (lm[15].y - cy)/scale)
        else:
            body_kp = [0.0] * DIMS_BODY
            if not hasattr(self, '_last_cx'):
                self._last_cx, self._last_cy, self._last_scale = 0.5, 0.5, 1.0
                self._last_nose = (0.0, -1.0)
                self._last_rw_n = (0.3, 0.5)
                self._last_lw_n = (-0.3, 0.5)

        # --- FACE ---
        face_kp = []
        if face_results.multi_face_landmarks:
            flags['face_detected'] = True
            fl = face_results.multi_face_landmarks[0].landmark
            cx, cy, scale = self._last_cx, self._last_cy, self._last_scale
            for idx in FACE_INDICES:
                p = fl[idx]
                face_kp.extend([(p.x - cx)/scale, (p.y - cy)/scale, p.z/scale])
            self._last_face_center_n = self._last_nose
        else:
            nx, ny = getattr(self, '_last_face_center_n', self._last_nose)
            face_kp = [val for _ in range(27) for val in [nx, ny, 0.0]]

        # --- HANDS ---
        lh = rh = None
        if hands_results.multi_hand_landmarks and hands_results.multi_handedness:
            cx, cy, scale = self._last_cx, self._last_cy, self._last_scale
            for hand_landmarks, handedness in zip(
                hands_results.multi_hand_landmarks,
                hands_results.multi_handedness
            ):
                label = handedness.classification[0].label
                flat = [(p.x - cx)/scale if i%3==0 else
                        (p.y - cy)/scale if i%3==1 else
                        p.z/scale
                        for p in hand_landmarks.landmark
                        for i, _ in enumerate([p.x, p.y, p.z])]
                flat = []
                for p in hand_landmarks.landmark:
                    flat.extend([(p.x - cx)/scale, (p.y - cy)/scale, p.z/scale])
                if label == "Right":
                    lh = flat[:DIMS_HAND]; flags['lh_detected'] = True
                else:
                    rh = flat[:DIMS_HAND]; flags['rh_detected'] = True

        if lh is None:
            lwx, lwy = self._last_lw_n
            lh = [val for _ in range(21) for val in [lwx, lwy, 0.0]]
        if rh is None:
            rwx, rwy = self._last_rw_n
            rh = [val for _ in range(21) for val in [rwx, rwy, 0.0]]

        result = np.array(body_kp + face_kp + lh + rh, dtype=np.float32)
        if len(result) > TOTAL_DIMS:
            result = result[:TOTAL_DIMS]
        elif len(result) < TOTAL_DIMS:
            result = np.pad(result, (0, TOTAL_DIMS - len(result)))
        return result, flags

    # =========================================================================
    # EMA SMOOTHING (giữ nguyên)
    # =========================================================================

    def _smooth_keypoints(self, keypoints_list, flags_list, alpha=0.6):
        if not keypoints_list:
            return keypoints_list
        smoothed = [keypoints_list[0].copy()]
        prev = keypoints_list[0].copy()
        for i in range(1, len(keypoints_list)):
            curr = keypoints_list[i]
            f = flags_list[i] if i < len(flags_list) else {}
            if not f.get('pose_detected', True):
                blended = alpha * curr + (1 - alpha) * prev
            else:
                blended = curr.copy()
                if not f.get('face_detected', True):
                    blended[IDX_FACE_START:IDX_LH_START] = (
                        alpha * curr[IDX_FACE_START:IDX_LH_START]
                        + (1 - alpha) * prev[IDX_FACE_START:IDX_LH_START])
                if not f.get('lh_detected', True):
                    blended[IDX_LH_START:IDX_RH_START] = (
                        alpha * curr[IDX_LH_START:IDX_RH_START]
                        + (1 - alpha) * prev[IDX_LH_START:IDX_RH_START])
                if not f.get('rh_detected', True):
                    blended[IDX_RH_START:] = (
                        alpha * curr[IDX_RH_START:]
                        + (1 - alpha) * prev[IDX_RH_START:])
            smoothed.append(blended)
            prev = blended
        return smoothed

    # =========================================================================
    # SAVE SEGMENT
    # =========================================================================

    def save_segment(self, segment_data, base_name, video_idx, seg_idx):
        file_name = f"{base_name}_v{video_idx:03d}_s{seg_idx:03d}"
        smoothed  = self._smooth_keypoints(
            segment_data['keypoints'], segment_data.get('flags', []), alpha=0.6)
        motion = np.array(smoothed)
        np.save(self.motion_dir / f"{file_name}.npy", motion)

        txt     = segment_data['text']
        content = f"{txt}#{txt}#0.0#0.0"
        with open(self.text_dir / f"{file_name}.txt", 'w', encoding='utf-8') as f:
            f.write(content)

        self.all_file_names.append(file_name)
        print(f"  💾 Saved: {file_name} | Frames: {len(motion)} | Text: {txt}")

    # =========================================================================
    # PROCESS VIDEO  ← PHẦN ĐƯỢC SỬA CHÍNH
    # =========================================================================

    def process_video(self, video_path, video_idx):
        print(f"\n🎬 Đang xử lý: {video_path.name}")
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            return

        for attr in ['_last_cx', '_last_cy', '_last_scale',
                     '_last_nose', '_last_rw_n', '_last_lw_n', '_last_face_center_n']:
            if hasattr(self, attr):
                delattr(self, attr)

        # Buffer lưu segment hiện tại
        current_buffer = {
            "text":                   "",
            "keypoints":              [],
            "flags":                  [],
            "frames_since_last_text": 0
        }

        segment_count = 0
        frame_idx     = 0
        pbar = tqdm(total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), desc="Frames")

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # --- OCR mỗi ocr_interval frame ---
            if frame_idx % self.ocr_interval == 0:
                detected_text = self.detect_text_regions(frame)
            else:
                detected_text = current_buffer["text"]

            # ================================================================
            # LOGIC PHÂN ĐOẠN CÓ FUZZY MATCHING
            # ================================================================

            if detected_text:
                same = is_same_sentence(
                    current_buffer["text"],
                    detected_text,
                    threshold=self.similarity_threshold
                )

                if same:
                    # ---- Cùng câu (kể cả OCR bị lỗi nhỏ) ----
                    # Cập nhật text nếu phiên bản mới "sạch" hơn
                    current_buffer["text"] = pick_better_text(
                        current_buffer["text"], detected_text)
                    current_buffer["frames_since_last_text"] = 0

                else:
                    # ---- Câu MỚI thực sự ----
                    if len(current_buffer["keypoints"]) > 15 and current_buffer["text"]:
                        self.save_segment(
                            current_buffer, video_path.stem, video_idx, segment_count)
                        segment_count += 1

                    current_buffer = {
                        "text":                   detected_text,
                        "keypoints":              [],
                        "flags":                  [],
                        "frames_since_last_text": 0
                    }

            else:
                # ---- Không detect được text ----
                current_buffer["frames_since_last_text"] += 1
                if current_buffer["frames_since_last_text"] > 45 and current_buffer["text"]:
                    if len(current_buffer["keypoints"]) > 15:
                        self.save_segment(
                            current_buffer, video_path.stem, video_idx, segment_count)
                        segment_count += 1
                    current_buffer = {
                        "text": "", "keypoints": [], "flags": [],
                        "frames_since_last_text": 0
                    }

            # ================================================================
            # KEYPOINT EXTRACTION
            # ================================================================
            if current_buffer["text"]:
                kp, flags = self.extract_openpose_keypoints(frame)
                current_buffer["keypoints"].append(kp)
                current_buffer["flags"].append(flags)

            frame_idx += 1
            pbar.update(1)
            if frame_idx % 500 == 0:
                gc.collect()

        # Flush buffer cuối
        if current_buffer["text"] and len(current_buffer["keypoints"]) > 15:
            self.save_segment(
                current_buffer, video_path.stem, video_idx, segment_count)

        cap.release()
        pbar.close()

    # =========================================================================
    # PROCESS PLAYLIST
    # =========================================================================

    def process_playlist(self, playlist_url):
        videos = self.download_playlist(playlist_url)
        if not videos:
            print("❌ Không tìm thấy video.")
            return

        for idx, video_path in enumerate(videos):
            check_pattern = f"_v{idx:03d}_"
            existing      = list(self.text_dir.glob(f"*{check_pattern}*.txt"))
            if existing:
                print(f"⏩ Video {idx} đã xử lý. Bỏ qua.")
                self.all_file_names.extend([f.stem for f in existing])
                try:
                    video_path.unlink()
                except Exception:
                    pass
                continue
            try:
                self.process_video(video_path, idx)
                video_path.unlink()
            except Exception as e:
                print(f"❌ Lỗi video {idx}: {e}")

        self.finalize()

    # =========================================================================
    # FINALIZE
    # =========================================================================

    def finalize(self):
        print("\n🏁 Đang hoàn tất...")
        if not self.all_file_names:
            self.all_file_names = [f.stem for f in self.text_dir.glob("*.txt")]
        if not self.all_file_names:
            print("❌ Không có dữ liệu nào.")
            return

        import random
        random.shuffle(self.all_file_names)
        total     = len(self.all_file_names)
        train_end = int(total * 0.9)
        val_end   = int(total * 0.95)

        splits = {
            "train.txt": self.all_file_names[:train_end],
            "val.txt":   self.all_file_names[train_end:val_end],
            "test.txt":  self.all_file_names[val_end:],
            "all.txt":   self.all_file_names
        }
        for name, files in splits.items():
            with open(self.base_dir / name, 'w') as f:
                f.write('\n'.join(files))

        print("📦 Đang nén...")
        shutil.make_archive("/kaggle/working/dataset_output", 'zip', self.base_dir)
        print("✅ Đã tạo: dataset_output.zip")


# =============================================================================
# MAIN
# =============================================================================

def main():
    PLAYLIST_URL = "https://www.youtube.com/playlist?list=PLUGvmK5KUAgtjWMAWJ0kwVRHrDLssJeke"

    processor = YouTubeToSignSAMProcessor(
        output_dataset="YOUTUBE_SIGN",
        fps=30,
        ocr_interval=8,
        similarity_threshold=0.82   # Tăng lên 0.90 nếu muốn strict hơn
    )
    processor.process_playlist(PLAYLIST_URL)


if __name__ == "__main__":
    main()

2026-03-29 15:16:22.135236: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774797382.569691      25 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774797382.677261      25 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774797383.656292      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774797383.656342      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774797383.656344      25 computation_placer.cc:177] computation placer alr

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
📥 Đang khởi tạo models...
 - Loading EasyOCR...


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1774797417.110840      92 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774797417.112348      86 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774797417.140255      90 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774797417.148650      86 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774797417.208147      83 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774797417.249164     

 - Loading YOLO...
✅ Khởi tạo hoàn tất! Vector size = 231 dims
   Similarity threshold = 0.82
📥 Đang tải playlist: https://www.youtube.com/playlist?list=PLUGvmK5KUAgtjWMAWJ0kwVRHrDLssJeke


ERROR: [youtube] BtFi2QI2CJg: This video is not available


...

  💾 Saved: 089_39HZLwEB2nA_v087_s026 | Frames: 208 | Text: Người điếc có thẻ khuyết tật thuộc nhóm đối tượng được miễn vé


Frames:  99%|█████████▉| 4914/4972 [07:33<00:06,  9.51it/s]

  💾 Saved: 089_39HZLwEB2nA_v087_s027 | Frames: 128 | Text: Chúc các bạn điếc có trải nghiệm mới về tàu điện!


Frames: 100%|██████████| 4972/4972 [07:38<00:00, 10.84it/s]


  💾 Saved: 089_39HZLwEB2nA_v087_s028 | Frames: 60 | Text: Cảm ơn mọi người đã theo dõi.

🏁 Đang hoàn tất...
📦 Đang nén...
✅ Đã tạo: dataset_output.zip
